In [ ]:
# Install dependencies
%pip install torch
%pip install -U transformers==4.57.1 trl==0.25.1 datasets==4.4.1
!pip install bitsandbytes
!pip install peft

In [2]:
import gc
import torch
torch.cuda.empty_cache()
gc.collect()

30

In [3]:
# Authenticate with HuggingFace
from google.colab import userdata
from huggingface_hub import login

In [4]:
# Set your HF_TOKEN in Colab secrets (key icon in sidebar)
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [5]:
# Generate the training dataset
!python generate_calendar_dataset.py

Generated 1360 examples
  Train: 1225
  Eval:  135
  Multi-step: 744
Saved to: calendar_training_data.jsonl


In [6]:
# Load Qwen2.5-1.5B
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-3B-Instruct"
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    attn_implementation="eager",
    dtype="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [7]:
import json
import datetime
from datasets import load_dataset

dataset = load_dataset("json", data_files="calendar_training_data.jsonl")["train"].shuffle(seed=42)

def sanitize_datetimes(obj):
    """Recursively convert HF-inferred datetime objects back to ISO strings."""
    if isinstance(obj, dict):
        return {k: sanitize_datetimes(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [sanitize_datetimes(v) for v in obj]
    elif isinstance(obj, datetime.datetime):
        # Cast back to the exact format your Swift app requires
        return obj.strftime("%Y-%m-%dT%H:%M")
    return obj

def apply_format_and_tokenize(sample):
    # 1. Sanitize the messages to strip out any Python datetime objects
    clean_messages = sanitize_datetimes(sample['messages'])

    # 2. Unwrap the tools (our fix from the previous step)
    hf_tools = [tool["function"] if "function" in tool else tool for tool in sample['tools']]

    # 3. Generate full conversation text using the cleaned messages
    full_text = tokenizer.apply_chat_template(
        clean_messages,
        tools=hf_tools,
        tokenize=False,
        add_generation_prompt=False
    )

    # 4. Tokenize
    encoded = tokenizer(full_text, add_special_tokens=False)
    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    # 5. Create labels array initialized to -100
    labels = [-100] * len(input_ids)

    # Qwen2.5 ChatML markers
    im_start = tokenizer.convert_tokens_to_ids("<|im_start|>")
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")

    in_assistant_turn = False

    # 6. Walk the tokens and unmask only the assistant's outputs
    for i, token_id in enumerate(input_ids):
        if token_id == im_start:
            decoded_chunk = tokenizer.decode(input_ids[i:i+3])
            if "assistant" in decoded_chunk:
                in_assistant_turn = True
                continue

        if in_assistant_turn:
            labels[i] = token_id

        if token_id == im_end and in_assistant_turn:
            in_assistant_turn = False

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "split": sample.get("metadata", "train"),
    }

processed_dataset = dataset.map(apply_format_and_tokenize)
train_dataset = processed_dataset.filter(lambda x: x['split'] == 'train')
eval_dataset = processed_dataset.filter(lambda x: x['split'] == 'eval')

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")
max_tokens = max(len(x['input_ids']) for x in processed_dataset) + 100
print(f"Max token count: {max_tokens}")

Train: 1225, Eval: 135
Max token count: 3537


In [8]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch

# Switch to the standard Trainer to bypass TRL's strict formatting assumptions
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model

output_dir = "/content/calendar-qwen"

args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,     # <-- FORCE EVAL TO BATCH SIZE 1
    eval_accumulation_steps=1,        # <-- OFFLOAD EVAL MEMORY TO CPU
    gradient_accumulation_steps=16,
    logging_strategy="steps",
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=50,
    save_strategy="epoch",
    learning_rate=8e-6,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    bf16=True,
    report_to="none"
)

base_model.config.pad_token_id = tokenizer.pad_token_id
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=base_model, padding=True)

base_model.enable_input_require_grads()

# 1. Define the LoRA adapter configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 2. Wrap your base model with the adapter
base_model = get_peft_model(base_model, peft_config)
base_model.print_trainable_parameters() # This will show you are only training ~1% of the model!

trainer = Trainer(
    model=base_model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)

trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("Training complete!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
50,0.386300,0.307762
100,0.146300,0.204959
150,0.089700,0.174043
200,0.070700,0.165337


Training complete!


In [9]:
from transformers import AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import whoami

# 1. Load the ORIGINAL base model
original_model_id = "Qwen/Qwen2.5-3B-Instruct" # <-- Replace if you used a different starting model
raw_model = AutoModelForCausalLM.from_pretrained(original_model_id, device_map="auto")

# 2. Load your trained LoRA adapter on top of it from your output directory
trained_model = PeftModel.from_pretrained(raw_model, output_dir)

# 3. Merge the adapter weights into the base weights (Crucial for Swift/iOS deployment!)
print("Merging weights... (this might take a minute)")
merged_model = trained_model.merge_and_unload()

# 4. Push the fully merged model to Hugging Face
username = whoami()['name']
hf_repo_id = f"{username}/qwen2.5-3b-calendar-agent"

print("Pushing to hub...")
merged_model.push_to_hub(hf_repo_id, create_repo=True, commit_message="Calendar agent fine-tune (merged)")
tokenizer.push_to_hub(hf_repo_id)

print(f"Successfully uploaded to: https://huggingface.co/{hf_repo_id}")

Merging weights... (this might take a minute)
Pushing to hub...
Successfully uploaded to: https://huggingface.co/amoghghadge/qwen2.5-3b-calendar-agent


In [ ]:
!pip install -U torch transformers==4.57.1 peft accelerate
!pip uninstall -y tensorflow

In [ ]:
!pip install ai-edge-torch-nightly --force-reinstall --timeout 300

In [ ]:
!pip install ai-edge-litert-nightly

In [2]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# --- STEP 1: DOWNLOAD ALREADY MERGED MODEL ---
print("Downloading merged model from Hugging Face...")
repo_id = "amoghghadge/qwen2.5-3b-calendar-agent"
merged_dir = "/content/merged-qwen"

# Load your completely merged model directly to CPU
merged_model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    device_map="cpu",
    dtype=torch.bfloat16
)

# Save the full PyTorch model locally for the LiteRT converter
os.makedirs(merged_dir, exist_ok=True)
merged_model.save_pretrained(merged_dir)

# Pull the tokenizer from your repo and save it
tokenizer = AutoTokenizer.from_pretrained(repo_id)

# (Optional but safe) Ensure the chat template is set correctly
base_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
tokenizer.chat_template = base_tokenizer.chat_template

tokenizer.save_pretrained(merged_dir)
print("Download and save complete!")

# --- STEP 2: CONVERT TO LITERTLM ---
import os
from ai_edge_torch.generative.examples.qwen import qwen
from ai_edge_torch.generative.utilities import converter
from ai_edge_torch.generative.utilities.export_config import ExportConfig
from ai_edge_torch.generative.layers import kv_cache

os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Full metadata WITH the tool-calling Jinja template baked in
llm_metadata = r"""start_token {
  token_ids {
    ids: 151643
  }
}
stop_tokens {
  token_ids {
    ids: 151645
  }
}
prompt_templates {
  user {
    prefix: "<|im_start|>user\n"
    suffix: "<|im_end|>\n"
  }
  model {
    prefix: "<|im_start|>assistant\n"
    suffix: "<|im_end|>\n"
  }
}
sampler_params {
  k: 40
  p: 0.95
  temperature: 1
}
llm_model_type {
  qwen2p5 {}
}
jinja_prompt_template: "{%- if tools %}\n    {{- \'<|im_start|>system\\n\' }}\n    {%- if messages[0][\'role\'] == \'system\' %}\n        {{- messages[0][\'content\'] }}\n    {%- else %}\n        {{- \'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.\' }}\n    {%- endif %}\n    {{- \"\\n\\n# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>\" }}\n    {%- for tool in tools %}\n        {{- \"\\n\" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- \"\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\\"name\\\": <function-name>, \\\"arguments\\\": <args-json-object>}\\n</tool_call><|im_end|>\\n\" }}\n{%- else %}\n    {%- if messages[0][\'role\'] == \'system\' %}\n        {{- \'<|im_start|>system\\n\' + messages[0][\'content\'] + \'<|im_end|>\\n\' }}\n    {%- else %}\n        {{- \'<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n\' }}\n    {%- endif %}\n{%- endif %}\n{%- for message in messages %}\n    {%- if (message.role == \"user\") or (message.role == \"system\" and not loop.first) or (message.role == \"assistant\" and not message.tool_calls) %}\n        {{- \'<|im_start|>\' + message.role + \'\\n\' + message.content + \'<|im_end|>\' + \'\\n\' }}\n    {%- elif message.role == \"assistant\" %}\n        {{- \'<|im_start|>\' + message.role }}\n        {%- if message.content %}\n            {{- \'\\n\' + message.content }}\n        {%- endif %}\n        {%- for tool_call in message.tool_calls %}\n            {%- if tool_call.function is defined %}\n                {%- set tool_call = tool_call.function %}\n            {%- endif %}\n            {{- \'\\n<tool_call>\\n{\"name\": \"\' }}\n            {{- tool_call.name }}\n            {{- \'\", \"arguments\": \' }}\n            {{- tool_call.arguments | tojson }}\n            {{- \'}\\n</tool_call>\' }}\n        {%- endfor %}\n        {{- \'<|im_end|>\\n\' }}\n    {%- elif message.role == \"tool\" %}\n        {%- if (loop.index0 == 0) or (messages[loop.index0 - 1].role != \"tool\") %}\n            {{- \'<|im_start|>user\' }}\n        {%- endif %}\n        {{- \'\\n<tool_response>\\n\' }}\n        {{- message.content }}\n        {{- \'\\n</tool_response>\' }}\n        {%- if loop.last or (messages[loop.index0 + 1].role != \"tool\") %}\n            {{- \'<|im_end|>\\n\' }}\n        {%- endif %}\n    {%- endif %}\n{%- endfor %}\n{%- if add_generation_prompt %}\n    {{- \'<|im_start|>assistant\\n\' }}\n{%- endif %}\n"
"""

litertlm_output_dir = '/content/litertlm'
os.makedirs(litertlm_output_dir, exist_ok=True)

metadata_path = os.path.join(litertlm_output_dir, 'base_llm_metadata.textproto')
with open(metadata_path, 'w') as f:
    f.write(llm_metadata)

# This uses the ALREADY MERGED model — no re-merge needed
pytorch_model = qwen.build_3b_model("/content/merged-qwen")

export_config = ExportConfig()
export_config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
export_config.mask_as_input = True

converter.convert_to_litert(
    pytorch_model,
    output_path=litertlm_output_dir,
    output_name_prefix="calendar-qwen25-3b",
    prefill_seq_len=512,       # faster prefill for large tool prompts
    kv_cache_max_len=4096,     # match base model context window
    quantize="dynamic_int8",
    export_config=export_config,
    hf_tokenizer_model_path=os.path.join("/content/merged-qwen", "tokenizer.json"),
    base_llm_metadata_path=metadata_path,
    output_format="litertlm",
)
print("Conversion complete!")

Download and save complete!
Conversion complete!


In [3]:
from google.colab import drive
import os

# 1. Mount your Google Drive (This will pop up a permission prompt)
drive.mount('/content/drive')

# 2. Create a folder in your Drive to hold the model
drive_folder = '/content/drive/MyDrive/qwen'
os.makedirs(drive_folder, exist_ok=True)

# 3. Copy the converted model from Colab's temporary storage to your Drive
# Note: Double check the exact filename output by the converter, but it should look like this:
!cp /content/litertlm/calendar-qwen25-3b_q8_ekv4096.litertlm /content/drive/MyDrive/qwen/

print("Successfully backed up to Google Drive!")

Mounted at /content/drive
Successfully backed up to Google Drive!


In [4]:
!ls -la /content/litertlm/*.litertlm

-rw-r--r-- 1 root root 3119464448 Mar 16 06:15 /content/litertlm/calendar-qwen25-3b_q8_ekv4096.litertlm
